# Oppitunti 10 - AI-agentit tuotannossa

Tässä oppitunnissa opit **tuotantomalleja** AI-agentteja varten käyttäen Microsoft Agent Frameworkia (Python). Käymme läpi:

- **Havaittavuus** — ajoituksen ja lokituksen lisääminen agentin vuorovaikutuksiin
- **Arviointi** — arvioivan agentin käyttäminen vastausten laadun pisteyttämiseen
- **Kustannusten hallinta** — strategioita tokenien optimointiin ja mallin valintaan

Skenaario on **matkatoimisto**, joka auttaa käyttäjiä suunnittelemaan matkoja, ja jonka päälle on lisätty valvonta ja arviointi.


## Asennus


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
import time
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())


## Tuotantoon liittyvät näkökohdat

AI-agenttien siirtäminen prototyypeistä tuotantoon vaatii huolellista huomiota kolmeen peruspilariin:

1. **Havaittavuus** — Tarvitset näkyvyyttä siihen, mitä agentti tekee, kuinka kauan se vie, ja mitä työkaluja se kutsuu. Ilman jäljitystä ja lokitusta tuotantovirheiden vianmääritys on lähes mahdotonta.

2. **Arviointi** — Automaattiset laatutarkastukset varmistavat, että agentin vastaukset pysyvät ajan myötä oikeina, täydellisinä ja hyödyllisinä. Arvioija-agentti voi pisteyttää vastaukset määriteltyjä kriteerejä vastaan.

3. **Kustannusten hallinta** — Tokenien käyttö vaikuttaa suoraan kustannuksiin. Strategiat kuten kehotteiden optimointi, mallin valinta ja välimuisti auttavat pitämään kulut hallinnassa ilman, että laatu kärsii.


## Havaittavan agentin rakentaminen

Määrittelemme matkustustyökalut ja kietomme agenttikutsun ajoituksen ympärille, jotta voimme seurata latenssia. Tuotannossa integroisit OpenTelemetryyn tai vastaavaan jäljitystaustajärjestelmään.


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## Arviointimallit

Yleinen tuotantokäytäntö on käyttää toista agenttia **arvioijana**. Arvioija pisteyttää pääagentin vastauksen ennalta määriteltyjen kriteerien, kuten kattavuuden, tarkkuuden ja hyödyllisyyden, perusteella.

Tämä mahdollistaa:
- Automaattiset laatuportit ennen kuin vastaukset saavuttavat käyttäjät
- Regressioiden havaitseminen, kun kehotteet tai mallit muuttuvat
- Agentin suorituskyvyn jatkuva seuranta ajan kuluessa


In [ ]:
evaluator = await provider.create_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## Kustannusten hallintastrategiat

Kustannusten hallinta on ratkaisevan tärkeää tuotannossa käytettävien tekoälyagenttien kohdalla. Tässä ovat keskeiset strategiat:

| Strategia | Kuvaus |
|---|---|
| **Kehoteoptimointi** | Pidä järjestelmäohjeet ytimekkäinä. Poista päällekkäinen konteksti vähentääksesi syötteen tokeneja. |
| **Mallin valinta** | Käytä pienempiä, edullisempia malleja (esim. GPT-4o-mini) yksinkertaisiin tehtäviin kuten luokitteluun tai tiedon poimintaan, ja varaa suuremmat mallit monimutkaista päättelyä varten. |
| **Välimuisti** | Tallenna työkalujen tulokset ja usein toistuvat kyselyt välimuistiin välttääksesi tarpeettomat API-kutsut. |
| **Token-budjetit** | Aseta `max_tokens`-rajat estääksesi odottamattoman pitkät vastaukset. |
| **Ryhmittely** | Ryhmittele useita käyttäjäkyselyjä yhdeksi API-kutsuksi aina kun mahdollista. |

Käytännössä monitasoinen lähestymistapa toimii hyvin: ohjaa suoraviivaiset pyynnöt nopealle, edulliselle mallille ja nosta vain monimutkaisemmat kyselyt tehokkaammalle mallille.


## Yhteenveto

Tässä oppitunnissa opit, miten:

1. **Lisätä havaittavuus** agenttien vuorovaikutuksiin ajoituksen ja lokituksen avulla, luoden perustan jäljitykselle ja valvonnalle.
2. **Arvioida agenttien vastaukset** automaattisesti käyttämällä arvioija-agenttia, joka pisteyttää täydellisyyden, tarkkuuden ja hyödyllisyyden.
3. **Hallita kustannuksia** kehotteiden optimoinnin, mallin valinnan, välimuistin ja token-budjettien avulla.

Nämä tuotantokäytännöt auttavat varmistamaan, että tekoälyagenttisi ovat luotettavia, mitattavissa ja kustannustehokkaita laajassa mittakaavassa.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
Vastuuvapauslauseke:
Tämä asiakirja on käännetty käyttämällä tekoälypohjaista käännöspalvelua Co-op Translator (https://github.com/Azure/co-op-translator). Vaikka pyrimme tarkkuuteen, huomioithan, että automaattiset käännökset voivat sisältää virheitä tai epätarkkuuksia. Alkuperäistä asiakirjaa sen alkuperäisellä kielellä on pidettävä ensisijaisena lähteenä. Tärkeiden tietojen osalta suosittelemme ammattimaista ihmiskäännöstä. Emme ole vastuussa mahdollisista väärinymmärryksistä tai virheellisistä tulkinnoista, jotka johtuvat tämän käännöksen käytöstä.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
